In [4]:
import openpyxl
import pandas as pd
import numpy as np

In [10]:
xlsx_path = r"C:\Users\sungy\Documents\GitHub\BMI_Lab\cdm_processor\[BMI Lab] ECG_conclusion_mapping_260504_박혜진.xlsx"
df = pd.read_excel(xlsx_path, sheet_name="mapping")

df.columns = np.where(df.iloc[0].isna(), df.columns, df.iloc[0])
df = df.drop(index=0).reset_index(drop=True)

new_columns = list(df.columns)
new_columns[3:7] = [f"{col}_1" for col in new_columns[3:7]]
new_columns[7:11] = [f"{col}_2" for col in new_columns[7:11]]
new_columns[11:15] = [f"{col}_3" for col in new_columns[11:15]]
df.columns = new_columns

print(df.columns)
print(df.shape)
display(df)

Index(['image_finding_source_value', 'frequency', '1:N 매핑',
       'value_as_concept_id_1', 'concept_name_1', 'domain_1', 'vocabulary_1',
       'value_as_concept_id_2', 'concept_name_2', 'domain_2', 'vocabulary_2',
       'value_as_concept_id_3', 'concept_name_3', 'domain_3', 'vocabulary_3'],
      dtype='object')
(11039, 15)


,image_finding_source_value,frequency,1:N 매핑,value_as_concept_id_1,concept_name_1,domain_1,vocabulary_1,value_as_concept_id_2,concept_name_2,domain_2,vocabulary_2,value_as_concept_id_3,concept_name_3,domain_3,vocabulary_3
0,Normal sinus rhythm,1319165.0,1.0,4276669,Normal sinus rhythm,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Normal ECG,882369.0,1.0,4276669,Normal sinus rhythm,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abnormal ECG,742922.0,1.0,320536,Electrocardiogram abnormal,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Otherwise normal ECG,277949.0,1.0,4276669,Normal sinus rhythm,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sinus bradycardia,269482.0,1.0,4171683,Sinus bradycardia,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11034,wiyh atrial bigeminy,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11035,wiyh marked sinus arrhythmia with premature at...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11036,wiyh strain,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11037,WPW pattern,1.0,1.0,4253363,Wolff-Parkinson-White pattern,Condition,SNOMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
dfs_to_combine = []

for i in range(1, 4):
    sub_df = df[[
        'image_finding_source_value', 
        'frequency', 
        f'value_as_concept_id_{i}', 
        f'concept_name_{i}', 
        f'domain_{i}', 
        f'vocabulary_{i}'
    ]].copy()
    
    sub_df.columns = [
        'source_value', 
        'frequency', 
        'concept_id', 
        'concept_name', 
        'domain', 
        'vocabulary'
    ]
    
    # 1:N 매핑이 안 되어서 ID 값이 비어있는(NaN) 행은 제외하고 리스트에 추가합니다.
    sub_df = sub_df.dropna(subset=['concept_id'])
    
    dfs_to_combine.append(sub_df)

final_mapping_df = pd.concat(dfs_to_combine, ignore_index=True)
final_mapping_df['concept_id'] = final_mapping_df['concept_id'].astype('Int64')
final_mapping_df['source_value'] = final_mapping_df['source_value'].astype(str).str.strip()
final_mapping_df.to_csv("ECG_conclusion_mapping.csv", index=False)

display(final_mapping_df)

,source_value,frequency,concept_id,concept_name,domain,vocabulary
0,Normal sinus rhythm,1319165.0,4276669,Normal sinus rhythm,Condition,SNOMED
1,Normal ECG,882369.0,4276669,Normal sinus rhythm,Condition,SNOMED
2,Abnormal ECG,742922.0,320536,Electrocardiogram abnormal,Condition,SNOMED
3,Otherwise normal ECG,277949.0,4276669,Normal sinus rhythm,Condition,SNOMED
4,Sinus bradycardia,269482.0,4171683,Sinus bradycardia,Condition,SNOMED
...,...,...,...,...,...,...
2085,"Unusual P axis, possible ectopic atrial tachyc...",1.0,441872,Supraventricular premature beats,Condition,SNOMED
2086,"Unusual P axis, possible ectopic atrial tachyc...",1.0,441872,Supraventricular premature beats,Condition,SNOMED
2087,Wide QRS rhythm with APB and VPB,1.0,4089462,Ventricular premature complex,Condition,SNOMED
2088,Wide QRS tachycardia with Premature supraventr...,1.0,441872,Supraventricular premature beats,Condition,SNOMED
